In [1]:
from ultralytics import YOLO
import torch
import yaml
from pathlib import Path
from datetime import datetime
import json
import pandas as pd
import matplotlib.pyplot as plt

## Конфиги

Главный конфиг обучения

In [ ]:
class TrainingConfig:
    # --- Модель ---
    MODEL_NAME = 'yolo26m.pt'

    
    # --- Данные ---
    DATA_YAML = '/wrk/data/full_data/FULL_DATA/data.yml'
    
    # --- Основные параметры обучения ---
    EPOCHS = 100
    
    BATCH_SIZE = 12
    
    IMG_SIZE = 640
    
    OPTIMIZER = 'MuSGD'         # ['auto', 'SGD', 'Adam', 'AdamW', 'MuSGD']
    
    LR0 = 0.01                  # [0.0001-0.1] Начальная скорость обучения
    
    LRF = 0.01                  # [0.001-0.1] Множитель конечного LR (final_lr = lr0 * lrf)
    
    MOMENTUM = 0.937            # [0.9-0.99] Momentum для SGD
    
    WEIGHT_DECAY = 0.0005       # [0.0001-0.001] L2 регуляризация
    
    WARMUP_EPOCHS = 3.0         # [0.0-10.0] Эпохи разогрева
    
    WARMUP_MOMENTUM = 0.8       # [0.0-0.9] Momentum во время warmup
    
    WARMUP_BIAS_LR = 0.1        # [0.0-1.0] LR для bias во время warmup
    
    # --- Scheduler (изменение LR во времени) ---
    COSINE = True               # [True/False]
    
    # --- Regularization ---
    DROPOUT = 0.0               # [0.0-0.5] Dropout rate
    
    # # --- Аугментации (дублируют YAML, можно переопределить) ---
    HSV_H = 0.015               # Вариация оттенка
    HSV_S = 0.7                 # Вариация насыщенности
    HSV_V = 0.4                 # Вариация яркости
    
    DEGREES = 0.0               # Вращение (градусы)
    TRANSLATE = 0.1             # Сдвиг
    SCALE = 0.5                 # Масштабирование
    SHEAR = 0.0                 # Сдвиг (shear)
    PERSPECTIVE = 0.0           # Перспектива
    FLIP_LR = 0.5               # Горизонтальный флип
    FLIP_UD = 0.0               # Вертикальный флип
    
    MOSAIC = 1.0                # Mosaic аугментация
    MIXUP = 0.1                 # Mixup аугментация
    COPY_PASTE = 0.0            # Copy-paste аугментация
    
    # --- Advanced настройки ---
    AMP = True                  # [True/False] Automatic Mixed Precision
    
    PRETRAIN = True             # [True/False] Использовать предобученные веса
    
    VERBOSE = True              # [True/False] Подробный вывод
    
    PATIENCE = 30               # [10-100] Эпох без улучшения до остановки
    
    # --- Сохранение ---
    PROJECT = '/wrk/YOLO/runs/new_data/detect'     # Папка для результатов
    NAME = 'yolo26_train_' + datetime.now().strftime('%Y%m%d_%H%M%S')

    EXIST_OK = False            # [True/False] Перезаписывать существующие
    SAVE = True                 # Сохранять чекпоинты
    SAVE_PERIOD = 50            # Сохранять каждые N эпох (-1 = только лучшую)
    
    # --- Валидация ---
    VAL = True                  # [True/False] Валидация во время обучения
    VAL_SPLIT = 0.0             # [0.0-1.0] Доля от train для валидации (если нет val, у меня есть)
    
    # --- Multi-GPU ---
    DEVICE = '0'                # ['0', '0,1,2,3', 'cpu', 'auto']
    
    # --- Multi-scale training  ---
    MULTI_SCALE = False         # [True/False] Обучение на разных масштабах. Очень нестабильно! Несколько раз вылетало обучение из-за деления на 0
    
    # --- Квантование (после обучения) ---
    EXPORT_FORMAT = 'onnx'      # ['onnx', 'torchscript', 'engine', 'tflite', 'openvino']

    EXPORT_INT8 = False         # [True/False] INT8 квантование

    EXPORT_HALF = False         # [True/False] FP16 экспорт
    
    CALC_F1 = True
    PLOT_PR_CURVES = True

Функция валидации

In [ ]:
def validate_model(model, cfg):
    metrics = model.val(
        data=cfg.DATA_YAML,
        batch=cfg.BATCH_SIZE,
        imgsz=cfg.IMG_SIZE,
        device=cfg.DEVICE,
    )
    
    print(f"\n📊 МЕТРИКИ:")
    print(f"  mAP50: {metrics.box.map50:.4f}")
    print(f"  mAP50-95: {metrics.box.map:.4f}")
    print(f"  Precision: {metrics.box.mp:.4f}")
    print(f"  Recall: {metrics.box.mr:.4f}")
    
    return metrics

Функция построения доп. графиков

In [9]:
def calculate_f1_and_pr_curves(model, data_yaml, save_dir):
    analytics_dir = Path(save_dir) / 'advanced_analytics'
    analytics_dir.mkdir(parents=True, exist_ok=True)
    
    metrics = model.val(
        data=data_yaml,
        save_json=True, 
        save_dir=str(analytics_dir),
        plots=False,
        verbose=False
    )

    precisions = metrics.box.mp 
    recalls = metrics.box.mr
    
    if (precisions + recalls) > 0:
        global_f1 = 2 * (precisions * recalls) / (precisions + recalls)
    else:
        global_f1 = 0.0
        
    print(f"   🏆 Global F1-Score (IoU=0.5): {global_f1:.4f}")

    class_names = model.names
    num_classes = len(class_names)
    
    f1_per_class = []

    print(f"   📋 Precision (mean): {precisions:.4f}")
    print(f"   📋 Recall (mean): {recalls:.4f}")
    
    report = {
        "global_f1_score": float(global_f1),
        "precision": float(precisions),
        "recall": float(recalls),
        "mAP50": float(metrics.box.map50),
        "mAP50-95": float(metrics.box.map)
    }
    
    with open(analytics_dir / 'f1_report.json', 'w') as f:
        json.dump(report, f, indent=2)
        
    if cfg.PLOT_PR_CURVES:

        model.val(
            data=data_yaml,
            save_dir=str(analytics_dir),
            plots=True,
            verbose=False
        )
        
        src_pr = Path(analytics_dir) / 'val_PR_curve.png'

        generated_plots = list(Path(analytics_dir).glob('**/*curve*.png'))
        if generated_plots:
            print(f"   ✓ PR-кривые сохранены: {generated_plots[0]}")
        else:
            print("   ⚠️ Не удалось найти файл с PR-кривыми автоматически.")

    return report

In [ ]:
def plot_custom_training_curves(results_csv_path, save_dir):
    print("\n📈 ПОСТРОЕНИЕ КАСТОМНЫХ ГРАФИКОВ...")
    
    if not Path(results_csv_path).exists():
        print(f"   ⚠️ Файл results.csv не найден: {results_csv_path}")
        return
    
    df = pd.read_csv(results_csv_path)
    plots_dir = Path(save_dir) / 'custom_plots'
    plots_dir.mkdir(parents=True, exist_ok=True)
    
    print(f"   📋 Доступные столбцы в results.csv:")
    for i, col in enumerate(df.columns):
        print(f"      {i}: {col}")
    
    prec_col = None
    rec_col = None
    
    for col in df.columns:
        if prec_col is None and 'precision' in col.lower():
            prec_col = col
        if rec_col is None and 'recall' in col.lower():
            rec_col = col
    
    if prec_col is None:
        for col in df.columns:
            if 'prec' in col.lower() and 'loss' not in col.lower():
                prec_col = col
                break
    
    if rec_col is None:
        for col in df.columns:
            if 'rec' in col.lower() and 'loss' not in col.lower():
                rec_col = col
                break
    
    train_box_loss = None
    val_box_loss = None
    
    for col in df.columns:
        if 'box_loss' in col.lower():
            if 'train' in col.lower() or 'train/' in col.lower():
                train_box_loss = col
            elif 'val' in col.lower() or 'val/' in col.lower():
                val_box_loss = col
    
    if train_box_loss is None:
        for col in df.columns:
            if 'box_loss' in col.lower():
                train_box_loss = col
                break
    
    if val_box_loss is None:
        for col in df.columns:
            if 'box_loss' in col.lower() and col != train_box_loss:
                val_box_loss = col
                break
    
    map50_col = None
    map95_col = None
    
    for col in df.columns:
        if 'map50' in col.lower() or 'map_50' in col.lower():
            map50_col = col
        if 'map' in col.lower() and '50' not in col.lower() and '95' not in col.lower():
            if map95_col is None:
                map95_col = col
    
    if prec_col and rec_col:
        print(f"   ✓ Найдено: precision='{prec_col}', recall='{rec_col}'")
        
        df['F1-Score'] = 2 * (df[prec_col] * df[rec_col]) / (df[prec_col] + df[rec_col] + 1e-6)
        
        fig, ax = plt.subplots(figsize=(14, 7))
        ax.plot(df['epoch'], df['F1-Score'], label='F1-Score', color='green', linewidth=2)
        
        if map50_col and map50_col in df.columns:
            ax.plot(df['epoch'], df[map50_col], label='mAP50', color='blue', linestyle='--')
        if map95_col and map95_col in df.columns:
            ax.plot(df['epoch'], df[map95_col], label='mAP50-95', color='purple', linestyle=':')
        
        ax.set_xlabel('Epoch', fontsize=12)
        ax.set_ylabel('Score', fontsize=12)
        ax.set_title('F1-Score & mAP Over Epochs', fontsize=14, fontweight='bold')
        ax.legend(loc='lower right')
        ax.grid(True, alpha=0.3)
        ax.set_ylim(0, 1.0)
        
        plt.tight_layout()
        plt.savefig(plots_dir / 'f1_score.png', dpi=150, bbox_inches='tight')
        plt.close()
        print(f"   ✓ F1 график: {plots_dir / 'f1_score.png'}")
    else:
        print(f"   ⚠️ Не найдены столбцы precision/recall. Пропускаем F1 график.")
        print(f"      prec_col={prec_col}, rec_col={rec_col}")
    
    if train_box_loss or val_box_loss:
        print(f"   ✓ Найдено: train_loss='{train_box_loss}', val_loss='{val_box_loss}'")
        
        fig, ax = plt.subplots(figsize=(14, 7))
        
        if train_box_loss and train_box_loss in df.columns:
            ax.plot(df['epoch'], df[train_box_loss], label='Train Box Loss', color='red', linewidth=2)
        if val_box_loss and val_box_loss in df.columns:
            ax.plot(df['epoch'], df[val_box_loss], label='Val Box Loss', color='orange', linewidth=2)
        
        for col in df.columns:
            if 'loss' in col.lower() and col not in [train_box_loss, val_box_loss]:
                if 'cls' in col.lower():
                    ax.plot(df['epoch'], df[col], label=f'Train Cls Loss', color='red', linestyle='--', alpha=0.5)
                elif 'dfl' in col.lower():
                    ax.plot(df['epoch'], df[col], label=f'Train DFL Loss', color='brown', linestyle=':', alpha=0.5)
        
        ax.set_xlabel('Epoch', fontsize=12)
        ax.set_ylabel('Loss', fontsize=12)
        ax.set_title('Training Loss Curves', fontsize=14, fontweight='bold')
        ax.legend(loc='upper right')
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(plots_dir / 'loss_curves.png', dpi=150, bbox_inches='tight')
        plt.close()
        print(f"   ✓ Loss график: {plots_dir / 'loss_curves.png'}")
    else:
        print(f"   ⚠️ Не найдены столбцы loss. Пропускаем Loss график.")
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    
    if train_box_loss and train_box_loss in df.columns:
        axes[0, 0].plot(df['epoch'], df[train_box_loss], label='Train Loss', color='red')
    if val_box_loss and val_box_loss in df.columns:
        axes[0, 0].plot(df['epoch'], df[val_box_loss], label='Val Loss', color='orange')
    axes[0, 0].set_title('Loss Curves', fontweight='bold')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    if map50_col and map50_col in df.columns:
        axes[0, 1].plot(df['epoch'], df[map50_col], label='mAP50', color='blue')
    if map95_col and map95_col in df.columns:
        axes[0, 1].plot(df['epoch'], df[map95_col], label='mAP50-95', color='purple')
    axes[0, 1].set_title('mAP Metrics', fontweight='bold')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    if prec_col and rec_col and 'F1-Score' in df.columns:
        axes[1, 0].plot(df['epoch'], df['F1-Score'], label='F1-Score', color='green')
    axes[1, 0].set_title('F1-Score', fontweight='bold')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].set_ylim(0, 1.0)
    
    lr_cols = [col for col in df.columns if 'lr' in col.lower()]
    for col in lr_cols:
        if col in df.columns:
            axes[1, 1].plot(df['epoch'], df[col], label=col)
    axes[1, 1].set_title('Learning Rate', fontweight='bold')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(plots_dir / 'summary_all_metrics.png', dpi=150, bbox_inches='tight')
    plt.close()
    print(f"   ✓ Сводный график: {plots_dir / 'summary_all_metrics.png'}")
    
    print(f"\n📁 Все графики сохранены в: {plots_dir}")

Функция экспорта модели

In [11]:
def export_model(model, cfg):
    """Экспорт модели для деплоя"""
    
    print("\n📤 ЭКСПОРТ МОДЕЛИ")
    
    best_weights = f"{cfg.PROJECT}/{cfg.NAME}/weights/best.pt"
    
    print(f"Экспорт из: {best_weights}")
    print(f"Формат: {cfg.EXPORT_FORMAT}")
    print(f"INT8: {cfg.EXPORT_INT8}")
    print(f"FP16: {cfg.EXPORT_HALF}")
    

    model.export(
        format=cfg.EXPORT_FORMAT,
        int8=cfg.EXPORT_INT8,
        half=cfg.EXPORT_HALF,
        dynamic=True,  # 🔧 Динамический размер входа
        simplify=True,  # 🔧 Упростить граф
    )
    
    print(f"\n✅ Модель экспортирована!")
    print(f"📁 Путь: {best_weights.replace('.pt', '.' + cfg.EXPORT_FORMAT)}")

## Обучение

Параметры обучения были взяты те же, на которых предобучалась yolo26m разработчиками

In [ ]:
def train():
    print("\n🚀 ЗАПУСК ОБУЧЕНИЯ YOLO (исправленная версия)")

    model = YOLO("/wrk/YOLO/yolo26m.pt")

    results = model.train(
        data="/wrk/data/full_data/FULL_DATA/data.yml",
        # === Основные ===
        epochs=300,       
        batch=16,
        imgsz=640,
        device=0,
        
        optimizer="muSGD",
        lr0=0.00038,
        lrf=0.882,
        momentum=0.948,      
        weight_decay=0.00027,
        warmup_epochs=0.99,  

        box=9.83,
        cls=0.65,
        dfl=0.96,  

        end2end=True, 
        
        # === Аугментации ===
        hsv_h=0.013,
        hsv_s=0.353,
        hsv_v=0.194,
        translate=0.275,      
        scale=0.95,           
        fliplr=0.304,
        mosaic=0.992,       
        mixup=0.427,      
        copy_paste=0.304,
        degrees=0,
        shear=0,
        bgr=0,  
        
        verbose=True,
        patience=40,
        close_mosaic=10,
        
        # === Сохранение ===
        project=cfg.PROJECT,
        name=cfg.NAME,
        exist_ok=cfg.EXIST_OK,
        save=cfg.SAVE,
        save_period=cfg.SAVE_PERIOD,
        
        # === Валидация ===
        val=cfg.VAL,
        multi_scale=False,
        
        # === Данные ===
        workers=8,
        cache=True,
        seed=42,
        
        plots=True,
        save_json=True,
        
    )
    
    return model, results

In [13]:
cfg = TrainingConfig()

model, results = train()

save_dir = Path(cfg.PROJECT) / cfg.NAME


🚀 ЗАПУСК ОБУЧЕНИЯ YOLO (исправленная версия)
New https://pypi.org/project/ultralytics/8.4.41 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.32 🚀 Python-3.11.14 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24564MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0, box=9.83, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.65, compile=False, conf=None, copy_paste=0.304, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/wrk/data/full_data/FULL_DATA/data.yml, degrees=0, deterministic=True, device=0, dfl=0.96, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=True, epochs=300, erasing=0.4, exist_ok=False, fliplr=0.304, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.013, hsv_s=0.353, hsv_v=0.194, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.00038, lrf=0.882, mask_ratio=4, max_det=300, mixup=0.427, mode=train, m

In [14]:
validate_model(model, cfg)

if cfg.CALC_F1:
    calculate_f1_and_pr_curves(model, "/wrk/data/full_data/FULL_DATA/data.yml", save_dir)
    
plot_custom_training_curves(save_dir / 'results.csv', save_dir)

Ultralytics 8.4.32 🚀 Python-3.11.14 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24564MiB)
YOLO26m summary (fused): 132 layers, 20,354,078 parameters, 0 gradients, 67.9 GFLOPs
val: Fast image access ✅ (ping: 1.5±0.1 ms, read: 34.5±14.4 MB/s, size: 82.3 KB)
val: Scanning /wrk/data/full_data/FULL_DATA/val/labels.cache... 3033 images, 1815 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 3033/3033 1.2Git/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 253/253 15.8it/s 16.1s<0.1s
                   all       3033       1704      0.945      0.955      0.976      0.778
                  ship        206        482      0.936      0.918      0.974      0.792
                 plane        319        388      0.934      0.912      0.945      0.704
            helicopter        466        519      0.906      0.958      0.959       0.76
                  buoy        165        176      0.977      0.945      0.991      0.865
       